# Generating one radius at a time

In [2]:
# Generate the initial CS2-H2 ab initio design at R = 40 in Molpro input format.
# After these 40 geometries are evaluated, the hysteresis workflow can decide the R = 30 additions.

from pathlib import Path
import importlib.util
import sys

import numpy

FUNCTIONS_PATH = Path("functions.py")
if not FUNCTIONS_PATH.exists():
    FUNCTIONS_PATH = Path("Performance Code/functions.py")

spec = importlib.util.spec_from_file_location("pes_functions", FUNCTIONS_PATH)
pes_functions = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = pes_functions
spec.loader.exec_module(pes_functions)

CS2_H2_INITIAL_RADIUS = 40.0
CS2_H2_INITIAL_POINT_COUNT = 40
CS2_H2_INITIAL_MOLPRO_INPUT_PATH = Path("CS2_H2_mb_avqz_R_40.00_initial_40.in")
CS2_H2_RESULT_TABLE_PATH = Path("cs2_h2_avqz_R_40.00_initial_40.txt")

# Placeholder matching the old template location. Replace once the production CS bond length is chosen.
CS_BOND_LENGTH_BOHR = 2.1944
H2_BOND_LENGTH_BOHR = 1.448736
GHOST_BASIS_LABEL = "H3"

angular_grid = pes_functions.make_angular_candidate_grid()
radius_data_template = numpy.column_stack([
    numpy.full(len(angular_grid), CS2_H2_INITIAL_RADIUS),
    angular_grid,
    numpy.zeros(len(angular_grid)),
])

required_point_indices = pes_functions.nearest_angle_indices(
    radius_data_template,
    pes_functions.ACQUISITION_REQUIRED_ANGLE_POINTS_DEGREES,
)
all_available_indices = numpy.arange(len(radius_data_template), dtype=int)
selected_point_indices = pes_functions.select_spread_indices(
    required_point_indices,
    CS2_H2_INITIAL_POINT_COUNT,
    all_available_indices,
    radius_data_template,
)

cs2_h2_r40_points = radius_data_template[selected_point_indices, :4]
cs2_h2_r40_points = cs2_h2_r40_points[numpy.lexsort((
    cs2_h2_r40_points[:, 3],
    cs2_h2_r40_points[:, 2],
    cs2_h2_r40_points[:, 1],
))]


def molpro_array(name, values, values_per_line=8):
    formatted_values = [f"{float(value):11.7f}" for value in values]
    chunks = [
        formatted_values[start:start + values_per_line]
        for start in range(0, len(formatted_values), values_per_line)
    ]
    if len(chunks) == 1:
        return f"{name}= [ " + ", ".join(chunks[0]) + "  ]"
    lines = [f"{name}= [ " + ", ".join(chunks[0])]
    for chunk in chunks[1:]:
        lines.append("         " + ", ".join(chunk))
    lines[-1] += "  ]"
    return "\n".join(lines)


th1_block = molpro_array("TH1all", cs2_h2_r40_points[:, 1])
th2_block = molpro_array("TH2all", cs2_h2_r40_points[:, 2])
phi_block = molpro_array("PHIall", cs2_h2_r40_points[:, 3])

molpro_input = f"""***,
memory,256.0,m
!GPRINT,orbital,basis;
rhh={H2_BOND_LENGTH_BOHR:.6f} bohr ! H2 bond length placeholder copied from previous template
rcs={CS_BOND_LENGTH_BOHR:.4f} bohr   ! TODO replace with production CS bond length
rhc=rhh/2;

! Initial CS2-H2 points generated by the hysteresis point-selection workflow

  !  theta 1

{th1_block}


  !  theta 2

{th2_block}

  !  phi 1
{phi_block}


k=0;


R={CS2_H2_INITIAL_RADIUS:.2f}
do j=1,#PHIall


th1=TH1all(j);
th2=TH2all(j);
phi=PHIall(j)

RC= [0 0 0] bohr;
RcmH2=[0 0 R] bohr;

! CS2 is rotated by theta1 with Oy

rS1 = [-sin(th1)*rcs, 0 , cos(th1)*rcs] bohr;
rS2 = [ sin(th1)*rcs, 0 ,-cos(th1)*rcs] bohr;

! H2 is first rotated by th2, then rotated along z by phi

rH1= [-cos(phi)*sin(th2)*rHC,  -sin(phi)*sin(th2)*rHC,  R+rHC*cos(th2)] bohr
rH2= [ cos(phi)*sin(th2)*rHC,   sin(phi)*sin(th2)*rHC,  R-rHC*cos(th2)] bohr

rcm=(rH1+rH2+rS1+rS2+rc)/4 bohr

k=k+1;

RR(k)=R;
TTH1(k)=th1;
TTH2(k)=th2;
PPHI(k)=phi;


ORIENT,NOORIENT;
geomtyp=xyz

bohr
geometry={{
6         ! number of atoms
CS2  H2  initial R=40 points
C1 ,0.0,0.0,0.0
S1 ,rS1(1),rS1(2),rS1(3)
S2 ,rS2(1),rS2(2),rS2(3)
H1 ,rH1(1),rH1(2),rH1(3)
H2 ,rH2(1),rH2(2),rH2(3)
H3 ,rcm(1),rcm(2),rcm(3)
! now without F12
}}
BASIS
DEFAULT=avqz
atom1={GHOST_BASIS_LABEL}
s,{GHOST_BASIS_LABEL},1.341100E-01,2.309000E-01;
c,1.1,1.000000E+00;
c,2.2,1.000000E+00;
p,{GHOST_BASIS_LABEL},1.468800E-01,2.707300E-01;
c,1.1,1.000000E+00;
c,2.2,1.000000E+00;
d,{GHOST_BASIS_LABEL},1.763400E-01;
c,1.1,1.000000E+00;

END


dummy,H3
{{hf;wf,40,1,0;}}
ehf_CS2H2=energy
{{ccsd(t)}}

eci_CS2H2=energy

dummy,H1,H2,H3
{{hf;wf,38,1,0;}}
ehf_CS2=energy
ccsd(t)
eci_CS2=energy

dummy,C1,S1,S2,H3
{{hf;wf,2,1,0;}}
ehf_H2=energy
{{ccsd(t)}}
eci_H2=energy

pot_hf=(ehf_CS2H2-ehf_CS2-ehf_H2)*tocm
pot_ci=(eci_CS2H2-eci_CS2-eci_H2)*tocm
v_corr=pot_ci-pot_hf


POTCI(k)=pot_ci
POTHF(k)=pot_hf
VCORR(k)=v_corr
Etot(k)=eci_CS2H2
ECS2(k)=eci_CS2
EH2(k)=eci_H2


table,RR,TTH1,TTH2,PPHI,POTCI,POTHF,VCORR,Etot,ECS2,EH2
format,(f8.3,f12.6,f12.6,f12.6,e16.8,e16.8,e16.8,e20.10,e20.10,e20.10)

!produce a table with results
save,{CS2_H2_RESULT_TABLE_PATH.name},new                 !save the table in file



enddo;


----
"""

CS2_H2_INITIAL_MOLPRO_INPUT_PATH.write_text(molpro_input)

print(f"Saved {len(cs2_h2_r40_points)} R=40 CS2-H2 Molpro geometries to {CS2_H2_INITIAL_MOLPRO_INPUT_PATH}")
cs2_h2_r40_points


Saved 40 R=40 CS2-H2 Molpro geometries to CS2_H2_mb_avqz_R_40.00_initial_40.in


array([[ 40.,   0.,   0.,   0.],
       [ 40.,   0.,   0.,  90.],
       [ 40.,   0.,   0., 165.],
       [ 40.,   0.,  60.,  75.],
       [ 40.,   0.,  90.,   0.],
       [ 40.,   0., 105., 105.],
       [ 40.,   0., 120., 165.],
       [ 40.,   0., 165.,   0.],
       [ 40.,   0., 165., 120.],
       [ 40.,   0., 180., 180.],
       [ 40.,  30.,  60., 180.],
       [ 40.,  30., 180.,  60.],
       [ 40.,  45.,  45.,  45.],
       [ 40.,  45.,  45., 120.],
       [ 40.,  45., 120.,  45.],
       [ 40.,  60., 120., 150.],
       [ 40.,  60., 180., 120.],
       [ 40.,  75.,   0.,  75.],
       [ 40.,  75.,  75.,   0.],
       [ 40.,  90.,   0.,   0.],
       [ 40.,  90.,   0., 165.],
       [ 40.,  90.,  60., 150.],
       [ 40.,  90.,  90.,  90.],
       [ 40.,  90., 150.,  60.],
       [ 40.,  90., 165.,   0.],
       [ 40.,  90., 180., 180.],
       [ 40., 120.,  45.,  45.],
       [ 40., 135.,   0.,  90.],
       [ 40., 135.,  90.,   0.],
       [ 40., 135.,  90., 180.],
       [ 4